# 08｜理解回测绩效指标：收益、年化、波动、回撤、夏普

这一节不是评估真实策略，而是用一组**手工可检查的小样本数据**理解绩效指标。

你要完成的目标：

1. 知道 `returns` 和 `equity` 分别是什么。
2. 能解释总收益率、CAGR、年化波动率、最大回撤、Sharpe Ratio。
3. 会调用 `scripts/performance_metrics.py` 生成统一指标表。
4. 明白这些指标什么时候会误导你。

> 学习提醒：指标不是用来证明策略好，而是用来逼你更冷静地看风险和收益。


## 1. 先区分两个输入：returns 与 equity

在回测里最常见的两条序列是：

```text
returns[t] = 本周期收益率
equity[t] = 从初始资金复利滚出来的净值曲线
```

例如某策略连续 4 天收益是：

```text
+1%, -2%, +3%, 0%
```

如果初始净值是 `1.0`，那么净值曲线不是简单相加，而是复利相乘：

```text
第1天：1.0 * (1 + 0.01)
第2天：1.0 * (1 + 0.01) * (1 - 0.02)
第3天：继续乘 (1 + 0.03)
```


In [8]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.performance_metrics import (
    annualized_volatility,
    cagr,
    drawdown_curve,
    equity_curve_from_returns,
    format_performance_summary,
    max_drawdown,
    performance_summary,
    sharpe_ratio,
    total_return,
)

originDf = pd.read_csv("../data/sample_etf_daily_long.csv")
originDf["date"] = pd.to_datetime(originDf["date"])
originDf = originDf.sort_values(["date"])


In [9]:
returns = originDf["close"].pct_change()

equity = equity_curve_from_returns(returns)

df = pd.DataFrame({
    "return": returns,
    "equity": equity,
})

df


,return,equity
0,NaN,1.000000
1,-0.002408,0.997592
2,-0.009555,0.988059
3,-0.005890,0.982240
4,0.012259,0.994281
...,...,...
75,-0.011793,0.941702
76,-0.007672,0.934477
77,-0.001825,0.932771
78,0.004303,0.936785


## 2. 总收益率 total_return

总收益率回答的是：**从开始到结束，一共赚/亏了多少比例？**

公式：

```text
total_return = 期末净值 / 期初净值 - 1
```

注意：它不关心中间过程。两个策略总收益一样，风险可能完全不同。


In [10]:
manual_total_return = equity.iloc[-1] / equity.iloc[0] - 1
module_total_return = total_return(equity)

print(f"手算总收益率: {manual_total_return:.4%}")
print(f"模块总收益率: {module_total_return:.4%}")


手算总收益率: -6.0305%
模块总收益率: -6.0305%


## 3. CAGR：复合年化收益率

CAGR 回答的是：**如果这段收益按复利折算成一年，大约是多少年化增长率？**

在日频回测中常用 `periods_per_year=252`，因为一年大约有 252 个交易日。

公式思想：

```text
cagr = 期末净值 ** (一年周期数 / 样本周期数) - 1
```

重要提醒：样本很短时，CAGR 会被严重放大，不要过度解读。


In [11]:
periods_per_year = 252
manual_cagr = equity.iloc[-1] ** (periods_per_year / len(equity)) - 1
module_cagr = cagr(equity, periods_per_year=periods_per_year)

print(f"手算 CAGR: {manual_cagr:.4%}")
print(f"模块 CAGR: {module_cagr:.4%}")


手算 CAGR: -17.7930%
模块 CAGR: -17.7930%


## 4. 年化波动率 annual_vol

年化波动率回答的是：**收益率上下波动有多剧烈？**

它不是对价格算标准差，而是对周期收益率算标准差：

```text
annual_vol = std(returns, ddof=1) * sqrt(periods_per_year)
```

这里 `ddof=1` 表示样本标准差，常用于历史样本估计。


In [12]:
manual_vol = returns.std(ddof=1) * np.sqrt(periods_per_year)
module_vol = annualized_volatility(returns, periods_per_year=periods_per_year)

print(f"手算年化波动率: {manual_vol:.4%}")
print(f"模块年化波动率: {module_vol:.4%}")


手算年化波动率: 11.4340%
模块年化波动率: 11.4340%


## 5. 回撤曲线与最大回撤

回撤回答的是：**从历史最高点跌下来多少？**

每一天的回撤：

```text
drawdown[t] = 当前净值 / 截至当前的历史最高净值 - 1
```

最大回撤就是整条回撤曲线里最小的那个值。

这对你的目标很重要：你追求的是回撤控制和慢复利，不是只看收益率。


In [13]:
df["running_peak"] = df["equity"].cummax()
df["drawdown"] = drawdown_curve(df["equity"])

display(df)
print(f"最大回撤: {max_drawdown(df['equity']):.4%}")


,return,equity,running_peak,drawdown
0,NaN,1.000000,1.000000,0.000000
1,-0.002408,0.997592,1.000000,-0.002408
2,-0.009555,0.988059,1.000000,-0.011941
3,-0.005890,0.982240,1.000000,-0.017760
4,0.012259,0.994281,1.000000,-0.005719
...,...,...,...,...
75,-0.011793,0.941702,1.008529,-0.066262
76,-0.007672,0.934477,1.008529,-0.073426
77,-0.001825,0.932771,1.008529,-0.075117
78,0.004303,0.936785,1.008529,-0.071137


最大回撤: -7.5117%


## 6. Sharpe Ratio：单位波动换来的收益

Sharpe Ratio 回答的是：**每承担一份波动，大约换来多少超额收益？**

简化版，不考虑无风险利率：

```text
sharpe = mean(returns) / std(returns, ddof=1) * sqrt(periods_per_year)
```

更完整版本会先扣除无风险利率：

```text
period_rf = (1 + annual_risk_free_rate) ** (1 / periods_per_year) - 1
excess_returns = returns - period_rf
sharpe = mean(excess_returns) / std(excess_returns, ddof=1) * sqrt(periods_per_year)
```

注意：Sharpe 假设波动能代表风险，但它不一定能充分描述尾部风险、流动性风险和极端回撤。


In [14]:
manual_sharpe_no_rf = returns.mean() / returns.std(ddof=1) * np.sqrt(periods_per_year)
module_sharpe_no_rf = sharpe_ratio(returns, periods_per_year=periods_per_year)
module_sharpe_with_rf = sharpe_ratio(
    returns,
    periods_per_year=periods_per_year,
    annual_risk_free_rate=0.02,
)

print(f"手算 Sharpe（不扣无风险利率）: {manual_sharpe_no_rf:.4f}")
print(f"模块 Sharpe（不扣无风险利率）: {module_sharpe_no_rf:.4f}")
print(f"模块 Sharpe（扣 2% 年化无风险利率）: {module_sharpe_with_rf:.4f}")


手算 Sharpe（不扣无风险利率）: -1.6782
模块 Sharpe（不扣无风险利率）: -1.6782
模块 Sharpe（扣 2% 年化无风险利率）: -1.8514


## 7. 一次性生成绩效汇总表

真实 Notebook 里不建议每次手写一遍公式。你应该把策略收益率和净值曲线交给模块统一计算。

这样做有三个好处：

1. 口径一致，不会今天一种公式、明天一种公式。
2. 更容易测试，避免手滑写错。
3. 后面比较策略和基准时，表格结构稳定。


In [15]:
summary = pd.DataFrame(
    {
        "example_strategy": performance_summary(
            returns,
            equity=equity,
            periods_per_year=periods_per_year,
            annual_risk_free_rate=0.0,
        )
    }
).T

summary


,total_return,cagr,annual_vol,max_drawdown,sharpe_ratio
example_strategy,-0.060305,-0.17793,0.11434,-0.075117,-1.678179


In [ ]:
format_performance_summary(summary)


## 8. 和一个基准做对比

评价策略不能只看自己。至少要和一个基准比较。

下面构造一个示例基准，不代表真实市场，只用于理解表格比较方式。


In [17]:
benchmark_returns = pd.Series(
    [0.005] * len(returns),
    index=returns.index,
    name="benchmark_ret",
)
benchmark_equity = equity_curve_from_returns(benchmark_returns)

comparison = pd.DataFrame(
    {
        "example_strategy": performance_summary(
            returns,
            equity=equity,
            periods_per_year=periods_per_year,
        ),
        "example_benchmark": performance_summary(
            benchmark_returns,
            equity=benchmark_equity,
            periods_per_year=periods_per_year,
        ),
    }
).T

format_performance_summary(comparison)


,total_return,cagr,annual_vol,max_drawdown,sharpe_ratio
example_strategy,-6.03%,-17.79%,11.43%,-7.51%,-1.68
example_benchmark,48.29%,251.44%,0.00%,0.00%,90936569479923232.00


## 9. 怎么写周复盘

这周如果还没有真实策略，复盘不要写“评估 Week05 策略”。更准确的写法是：

```text
本周我用示例收益率和净值数据理解了绩效指标，并确认 performance_metrics.py 可以统一计算指标表。
```

可以回答这几个问题：

1. `returns` 和 `equity` 的区别是什么？
returns 是“每一期赚了多少”，比如日收益率 0.01 表示这一天涨了 1%。
equity 是“累计到现在一共变成多少”，也就是净值曲线，比如从 1.0 变成 1.01、再变成 0.99、再变成 1.03。
简单说：returns 是增量，equity 是累计结果。

2. 总收益率和 CAGR 为什么不是一回事？
total_return = final / initial - 1
cagr = final ** (periods_per_year / periods) - 1
3. 年化波动率为什么要对收益率算标准差？
因为波动率本质上是在看收益率“围绕平均值上下晃得有多厉害”。
标准差正是最常用的离散程度指标，所以拿它来量化收益的稳定性很自然。
再乘上 sqrt(年周期数)，就把日波动率、周波动率、月波动率统一到年化口径。
所以年化波动率不是在看涨跌方向，而是在看“起伏大小”。

4. 最大回撤为什么对实盘心理压力很重要？
因为它描述的是“从历史最高点跌下来最深跌了多少”。
人真正难受的，通常不是年化收益低一点，而是账户先涨后大幅回撤。
比如策略年化很好，但中途回撤 -35%，很多人会在最难熬的时候砍掉它。
所以最大回撤直接影响你能不能拿得住策略，这就是它很重要的原因。

5. Sharpe 高是否一定代表策略好？为什么不一定？
可能隐藏了尾部风险，比如平时很稳，偶尔大亏一次。
可能依赖特定市场环境，换个阶段就失效。
可能交易成本、滑点、容量一算进去就不好看。
可能回测样本太短，Sharpe 被偶然抬高。

6. 等以后有真实策略时，我会如何比较策略和基准？
收益：总收益、CAGR
风险：波动率、最大回撤
风险调整后收益：Sharpe、Sortino
相对基准：超额收益、是否长期跑赢基准
稳定性：胜率、回撤恢复速度、月度/年度表现分布


## 10. 本节边界

当前 Notebook 只用于理解指标，不代表完整回测评价。

未来评估真实策略时，还需要继续检查：

- 手续费和滑点；
- 信号和持仓是否 `shift(1)`，有没有偷看未来；
- 是否和合理基准比较；
- 样本内/样本外是否分开；
- 参数是否稳健；
- 收益是否集中来自少数极端日期。
